In [ ]:
from __future__ import annotations

import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.interpolate import UnivariateSpline

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.tuning_utils as tut

def vprint(msg: str) -> None:
    if VERBOSE:
        print(msg)


def pca_subspace_basis(X_u_by_img: np.ndarray, d: int, center: bool = True, eps: float = 1e-12) -> np.ndarray:
    """
    Return an orthonormal PCA basis in unit space for one timepoint.

    Args:
        X_u_by_img: Matrix with shape (units, images).
        d: Requested number of principal directions.
        center: Mean-center each unit across images before SVD.
        eps: Singular value threshold for effective rank.

    Returns:
        Basis matrix with shape (units, d_eff).
    """
    X = X_u_by_img.astype(np.float64, copy=False)
    if center:
        X = X - np.nanmean(X, axis=1, keepdims=True)

    X = np.nan_to_num(X, nan=0.0)
    U, S, _ = np.linalg.svd(X, full_matrices=False)

    rank_eff = int(np.sum(S > eps))
    d_eff = int(min(d, rank_eff, U.shape[1]))
    if d_eff == 0:
        return np.zeros((U.shape[0], 0), dtype=np.float64)
    return U[:, :d_eff]


def principal_angles(Q1: np.ndarray, Q2: np.ndarray) -> np.ndarray:
    """
    Return principal angles between two orthonormal subspaces.
    """
    if Q1.size == 0 or Q2.size == 0:
        return np.array([], dtype=np.float64)

    singular_vals = np.linalg.svd(Q1.T @ Q2, compute_uv=False)
    singular_vals = np.clip(singular_vals, 0.0, 1.0)
    return np.arccos(singular_vals)


def pca_spectrum(X_u_by_img: np.ndarray, center: bool = True):
    # compute singular values only
    X = X_u_by_img.astype(np.float64, copy=False)

    if center:
        X = X - np.nanmean(X, axis=1, keepdims=True)

    X = np.nan_to_num(X, nan=0.0)

    # full_matrices=False keeps this compact
    _, S, _ = np.linalg.svd(X, full_matrices=False)

    # eigenvalues of covariance ∝ S^2
    var = S**2
    var_ratio = var / np.sum(var)

    return var, var_ratio

In [ ]:
# MAIN IDEA: test for cross-validated subspace rotation
# HYPOTHESIS: transient subspace rotations that depend on location/scale on the natural image manifold
# CONCRETE: represent each time bin by the top-d PCs of the image-response matrix, then measure how much those subspaces rotate across time using principal angles

# (units, time, images, reps)
# images = either (1) local/top-k manifold scale or (2) all images


# ROI format:
# - 4-part UID: SesIdx.RoiIndex.AREALABEL.Categoty (e.g., 18.19.Unknown.F)
# - 3-part ROI: RoiIndex.AREALABEL.Categoty (e.g., 19.Unknown.F)
ROI_TARGET = "19.Unknown.F" # 19.Unknown.F
BIN_SIZE_MS = 20
ALPHA = 0.05
D_PCS = 50
BASE_SL = slice(0, 50)
POST_SL = slice(160, 200)
RANDOM_STATE = 0
VERBOSE = True

MINS_PATH = Path("./../../datasets/NNN/face_mins.pkl")
OUTPUT_PATH = Path.home() / "Downloads" / "rotation_19_Unknown_F.png"

In [ ]:
roi_key = ROI_TARGET.split(".")
if len(roi_key) == 4:
    roi_label = f"{roi_key[2]}_{int(roi_key[1])}_{roi_key[3]}"
else:
    roi_label = f"{roi_key[1]}_{int(roi_key[0])}_{roi_key[2]}"

with MINS_PATH.open("rb") as f:
    mins = pickle.load(f)

if roi_label not in mins:
    raise ValueError(f"{roi_label} not found in {MINS_PATH}")

local_k = int(mins[roi_label][0])
vprint(f"ROI label: {roi_label}")
vprint(f"Local scale k: {local_k}")

raster_4d = nu.significant_trial_raster(roi_uid=ROI_TARGET, alpha=ALPHA, bin_size_ms=BIN_SIZE_MS)
vprint(f"Responsive trial raster shape: {raster_4d.shape}")

split_a = raster_4d[:, :, :, 0::2]
split_b = raster_4d[:, :, :, 1::2]
psth_A = np.nanmean(split_a, axis=3)
psth_B = np.nanmean(split_b, axis=3)

if psth_A.shape != psth_B.shape:
    raise ValueError(f"A/B shape mismatch: A={psth_A.shape}, B={psth_B.shape}")

vprint(f"PSTH A shape: {psth_A.shape}")
vprint(f"PSTH B shape: {psth_B.shape}")

In [ ]:
trial_avg = np.nanmean(raster_4d, axis=3)
image_order = tut.rank_images_by_response(trial_avg)
rng = np.random.default_rng(RANDOM_STATE)

image_sets = {
    f"Local ({local_k})": image_order[:local_k],
    f"Global ({trial_avg.shape[2]})": np.arange(trial_avg.shape[2]),
    f"Random ({local_k})": rng.choice(trial_avg.shape[2], size=local_k, replace=False),
}

T = psth_A.shape[1]

set_rotation = {}
for label, image_idx in image_sets.items():
    QA = []
    QB = []
    for t in range(T):
        QA.append(pca_subspace_basis(psth_A[:, t, image_idx], d=D_PCS))
        QB.append(pca_subspace_basis(psth_B[:, t, image_idx], d=D_PCS))

    sim_cv = np.full((T, T), np.nan, dtype=np.float64)
    for t1 in range(T):
        for t2 in range(T):
            ang1 = principal_angles(QA[t1], QB[t2])
            ang2 = principal_angles(QB[t1], QA[t2])
            s1 = float(np.mean(np.cos(ang1) ** 2))
            s2 = float(np.mean(np.cos(ang2) ** 2))
            sim_cv[t1, t2] = np.nanmean([s1, s2])
    set_rotation[label] = sim_cv

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# X: (units, time, images) trial-averaged
# order: image indices sorted by preference (len = n_images)
# scale: k for top-k / random-k
# Example:
# X = tut.trial_averaged_psth(dat, ROI)
# order = tut.rank_images_by_response(X)
# scale = SCALES[ROI]

def top_eigvals_over_time(X, idx, n_pcs=10, center_across_images=True):
    """
    Returns L: (n_pcs, T) where L[p,t] is eigenvalue for PC p at time t,
    computed from covariance across images at that time.
    """
    U, T, K = X[:, :, idx].shape
    L = np.full((n_pcs, T), np.nan, dtype=np.float64)

    for t in range(T):
        M = X[:, t, idx]  # (units, K)
        if center_across_images:
            M = M - np.mean(M, axis=1, keepdims=True)

        # eigenvalues of (M M^T)/(K-1) are s^2/(K-1) where s are singular values of M
        # use SVD on (units x K), take top n_pcs singular values
        s = np.linalg.svd(M, full_matrices=False, compute_uv=False)
        lam = (s**2) / max(K - 1, 1)

        L[:, t] = lam[:n_pcs]

    return L

def plot_pc_lambda_heatmap(L, title, ax=None, vmax=1, log_scale=False):
    if ax is None:
        ax = plt.gca()

    Z = np.log1p(L) if log_scale else L

    im = ax.imshow(
        Z,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        vmax=vmax
    )
    ax.set_title(title)
    ax.set_xlabel("time bin")
    ax.set_ylabel("PC index")
    ax.set_yticks(range(L.shape[0]))
    ax.set_yticklabels([f"PC{i+1}" for i in range(L.shape[0])])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=("log1p(λ)" if log_scale else "λ"))

# ---- build indices for three conditions ----
k = local_k
n_images = trial_avg.shape[2]

idx_top = np.array(image_order[:k], dtype=int)
idx_all = np.arange(n_images, dtype=int)
idx_rand = rng.choice(n_images, size=k, replace=False)  # or rng.choice(order, size=k, replace=False)

# ---- compute lambdas over time ----
n_pcs = 20
L_top  = top_eigvals_over_time(trial_avg, idx_top,  n_pcs=n_pcs, center_across_images=True)
L_all  = top_eigvals_over_time(trial_avg, idx_all,  n_pcs=n_pcs, center_across_images=True)
L_rand = top_eigvals_over_time(trial_avg, idx_rand, n_pcs=n_pcs, center_across_images=True)

# ---- plot heatmaps (one per condition) ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
plot_pc_lambda_heatmap(L_top,  f"Top-{k}",   ax=axes[0], vmax = 0.015, log_scale=True)
plot_pc_lambda_heatmap(L_all,  "All images", ax=axes[1], vmax = 0.015, log_scale=True)
plot_pc_lambda_heatmap(L_rand, f"Random-{k}",ax=axes[2], vmax = 0.015, log_scale=True)

plt.tight_layout()
plt.show()

In [ ]:
def k_to_reach_evar_by_time(X_u_t_img, thresh=0.8, center=True, eps=1e-12):
    """
    X_u_t_img: (units, time, images)
    returns df with columns: t, k
    """
    X = np.asarray(X_u_t_img, dtype=np.float64)
    U, T, I = X.shape

    ks = np.full(T, np.nan, dtype=float)

    for t in range(T):
        Xt = X[:, t, :]

        if center:
            Xt = Xt - np.nanmean(Xt, axis=1, keepdims=True)

        Xt = np.nan_to_num(Xt, nan=0.0)

        # singular values -> variance ~ S^2
        _, S, _ = np.linalg.svd(Xt, full_matrices=False)
        S = S[S > eps]
        if S.size == 0:
            continue

        var = S**2
        cum = np.cumsum(var) / np.sum(var)

        # smallest k s.t. cum[k-1] >= thresh
        k = int(np.searchsorted(cum, thresh) + 1)
        ks[t] = k

    return pd.DataFrame({'t': np.arange(T), 'k': ks})


# ---- compute for the three image sets ----
dfs = []
for label, image_idx in image_sets.items():
    df_k = k_to_reach_evar_by_time(trial_avg[:, :, image_idx], thresh=0.8, center=True)
    df_k['set'] = label
    dfs.append(df_k)

df_plot = pd.concat(dfs, ignore_index=True)

# ---- plot ----
fig, ax = plt.subplots(figsize=(6, 3))
sns.lineplot(data=df_plot, x='t', y='k', hue='set', ax=ax)

ax.set_xlabel('time bin')
ax.set_ylabel('k to reach 80% evar')
ax.set_title('time-resolved dimensionality (k80)')
ax.set_ylim(top=30)

sns.despine(fig=fig, trim=True, offset=5)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for col, (label, sim_cv) in enumerate(set_rotation.items()):
    ax_hm = axes[0, col]
    sns.heatmap(sim_cv, square=True, cmap=sns.color_palette('Greys', as_cmap=True), ax=ax_hm, cbar=False)
    ax_hm.set_xlabel("time")
    ax_hm.set_ylabel("time")

    baseline_mean = float(np.nanmean(sim_cv[BASE_SL, BASE_SL]))
    post_mean = float(np.nanmean(sim_cv[POST_SL, POST_SL]))
    percent_change = (post_mean - baseline_mean)/baseline_mean
    ax_hm.set_title(f'{label} | {percent_change*100:.01f}%')

    ax_bar = axes[1, col]
    ax_bar.bar(["baseline", "post"], [baseline_mean, post_mean], color=["0.6", "0.2"])
    ax_bar.set_title(f"Mean similarity: {label}")
    ax_bar.set_ylabel("similarity")
    ax_bar.text(0, baseline_mean, f"{baseline_mean:.3f}", ha="center", va="bottom")
    ax_bar.text(1, post_mean, f"{post_mean:.3f}", ha="center", va="bottom")

In [ ]:
from scipy.interpolate import UnivariateSpline
from scipy.stats import chi2


def gaussian_loglik(rss, n):
    # gaussian ll at mle sigma^2 = rss/n
    sigma2 = rss / n
    return -0.5 * n * (np.log(2 * np.pi * sigma2) + 1)


def aic_bic(ll, n, k):
    aic = 2 * k - 2 * ll
    bic = k * np.log(n) - 2 * ll
    return aic, bic


fig, axes = plt.subplots(1, 3, figsize=(15, 3), constrained_layout=True)

for col, (label, sim_cv) in enumerate(set_rotation.items()):
    ax = axes[col]
    y = np.asarray(np.diag(sim_cv), dtype=float)
    x = np.arange(y.size, dtype=float)

    m = np.isfinite(y)
    x0, y0 = x[m], y[m]

    sns.lineplot(x=x, y=y, ax=ax)
    ax.set_title(label)

    if y0.size < 5:
        continue

    # -------------------
    # linear fit
    # -------------------
    slope, intercept, *_ = stats.linregress(x0, y0)
    yhat_lin = intercept + slope * x0
    rss_lin = float(np.sum((y0 - yhat_lin) ** 2))
    k_lin = 3  # slope + intercept + variance
    ll_lin = float(gaussian_loglik(rss_lin, y0.size))
    aic_lin, bic_lin = aic_bic(ll_lin, y0.size, k_lin)

    # -------------------
    # smoothing spline
    # -------------------
    sp = UnivariateSpline(x0, y0, k=3, s=None)  # default smoothing
    yhat_sp = sp(x0)
    rss_sp = float(np.sum((y0 - yhat_sp) ** 2))
    k_sp = len(sp.get_coeffs()) + 1  # coeffs + variance (df proxy)
    ll_sp = float(gaussian_loglik(rss_sp, y0.size))
    aic_sp, bic_sp = aic_bic(ll_sp, y0.size, k_sp)

    # -------------------
    # lrt (approx; df is proxy for smoothing spline)
    # -------------------
    df_lrt = max(1, int(k_sp - k_lin))
    lrt_stat = max(0.0, 2.0 * (ll_sp - ll_lin))
    lrt_p = float(1.0 - chi2.cdf(lrt_stat, df_lrt))

    # plot fits
    ax.plot(
        x0,
        yhat_lin,
        linestyle='--',
        linewidth=2,
        label=f'lin bic={bic_lin:.1f} ll={ll_lin:.1f}',
    )
    ax.plot(
        x0,
        yhat_sp,
        linestyle=':',
        linewidth=2,
        label=f'spl bic={bic_sp:.1f} ll={ll_sp:.1f} lrt p={lrt_p:.2g}',
    )
    ax.legend(frameon=False, fontsize=8)

plt.show()